# Molecular Descriptor Calculation

## Import Libraries

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator



## Import Clean Dataset


In [2]:
df=pd.read_csv("../data/output/pIC50LibA.csv")
df

,Smile,ID,IC50(nM),pIC50
0,C#CCOc1cnc(/C(F)=C/c2ccc(F)c([C@]3(C)CS(=O)(=O...,CHEMBL6010325,3.00,8.522879
1,C#CCOc1cnc(/C(F)=C/c2ccc(F)c([C@]34CN(C(=O)OC(...,CHEMBL5905447,2.62,8.581699
2,CC#Cc1cncc(-c2ccc3c(c2)[C@@]2(COC(N)=N2)c2cc(-...,C27H18FN5O2,0.43,9.366532
3,CC#Cc1cncc(-c2ccc3c(c2)[C@@]2(COC(N)=N2)c2cc(C...,C27H22N4O3,0.46,9.337242
4,CC#Cc1cncc(-c2ccc3c(c2)[C@@]2(COC(N)=N2)c2cc(C...,C27H22N4O3,0.31,9.508638
...,...,...,...,...
172,NC1=N[C@@]2(CO1)c1cc(NC(=O)c3ccc(Cl)cn3)ccc1Oc...,C23H18ClN5O3,3.60,8.443697
173,NC1=N[C@@]2(CO1)c1cc(NC(=O)c3ccc(Cl)cn3)ccc1Oc...,C25H22ClN5O4,0.57,9.244125
174,NC1=N[C@@]2(CO1)c1cc(NC(=O)c3ccc(Cl)cn3)ccc1Oc...,C20H13Cl2N5O3,21.00,7.677781
175,NC1=N[C@@]2(c3cc(-c4cccnc4F)ccc3Oc3c(F)cc([N+]...,CHEMBL4109073,0.60,9.221849


## Import Descriptor Names


In [33]:
def descriptor_names():
    return [name for name,func in Descriptors._descList]

## Import Descriptor Functions


In [8]:
def get_descriptors_functions():
    return Descriptors._descList

##  Basic Descriptors

In [3]:
def calculate_basic_descriptors(df, smiles_column="Smile"):
    mols=df[smiles_column].apply(Chem.MolFromSmiles)
    dsc=[]
    for mol in mols:
        dsc.append({'NumAtoms':mol.GetNumAtoms(),
                    'MW':Descriptors.MolWt(mol),
                    'HBA':Descriptors.NumHAcceptors(mol),
                    'HBD':Descriptors.NumHDonors(mol),
                    'logP':Descriptors.MolLogP(mol),
                    'TPSA':Descriptors.TPSA(mol),
                    'RotatableBonds':Descriptors.NumRotatableBonds(mol),
                    'NumAromaticRing':Descriptors.NumAromaticRings(mol)})
    dc=pd.DataFrame(dsc)
    return dc

In [4]:
dc=calculate_basic_descriptors(df)
dc

,NumAtoms,MW,HBA,HBD,logP,TPSA,RotatableBonds,NumAromaticRing
0,33,472.517,7,1,3.0188,107.53,5,2
1,36,513.570,8,1,3.8289,102.93,6,2
2,35,463.472,7,1,4.4103,95.51,2,4
3,34,450.498,7,1,4.0129,91.85,2,3
4,34,450.498,7,1,4.0129,91.85,2,3
...,...,...,...,...,...,...,...,...
172,32,447.882,7,2,3.9539,111.72,3,3
173,35,491.935,8,2,3.9705,120.95,3,3
174,30,442.262,7,2,3.7299,111.72,2,3
175,36,485.474,6,1,3.5423,90.56,2,4


##  Extended Descriptors

In [9]:
def calculate_extended_descriptors(df,smiles_column="Smile"):
    des_list=get_descriptors_functions()
    mols=df[smiles_column].apply(Chem.MolFromSmiles)
    descriptors=[]
    
    for mol in mols:
        if mol is None:
           descriptors.append({name: np.nan for name, _ in des_list})
           continue
        row={}
        for name, func in des_list:
           try:
            row[name]=func(mol)
           except Exception:
            row[name]=np.nan

        descriptors.append(row)
        
    descriptors_df=pd.DataFrame(descriptors) 
    return descriptors_df

In [10]:
ds=calculate_extended_descriptors(df)
ds

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,14.803745,14.803745,0.007250,-3.652177,0.670633,23.333333,472.517,450.341,472.138068,172,...,0,0,1,1,0,0,0,0,0,0
1,15.242830,15.242830,0.019417,-1.105593,0.585889,21.555556,513.570,488.370,513.164617,188,...,1,0,0,1,0,0,0,0,0,0
2,13.781613,13.781613,0.087412,-0.943911,0.349535,17.371429,463.472,445.328,463.144453,170,...,0,0,0,0,0,0,0,0,0,0
3,6.278086,6.278086,0.157097,-0.828326,0.593360,20.147059,450.498,428.322,450.169191,168,...,0,0,0,0,0,0,0,0,0,0
4,6.284645,6.284645,0.161788,-0.818760,0.593360,20.147059,450.498,428.322,450.169191,168,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,12.654788,12.654788,0.118877,-0.862262,0.628253,20.593750,447.882,429.738,447.109817,160,...,0,0,0,0,0,0,0,0,0,0
173,12.737734,12.737734,0.104659,-0.907622,0.568150,20.885714,491.935,469.759,491.136032,178,...,0,0,0,0,0,0,0,0,0,0
174,12.555442,12.555442,0.056979,-0.959484,0.585416,18.666667,442.262,429.158,441.039545,150,...,0,0,0,0,0,0,0,0,0,0
175,15.651898,15.651898,0.048905,-1.297290,0.346568,19.722222,485.474,466.322,485.153207,178,...,0,0,0,0,0,0,0,0,0,0


## Descriptor Information


In [40]:
ds.shape

(177, 217)

In [41]:
ds.info()

<class 'pandas.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Columns: 217 entries, MaxAbsEStateIndex to fr_urea
dtypes: float64(107), int64(110)
memory usage: 300.2 KB


In [44]:
ds.describe().T

,count,mean,std,min,25%,50%,75%,max
MaxAbsEStateIndex,177.0,12.487045,2.986793,6.167710,12.705712,13.782625,14.101339,15.651898
MaxEStateIndex,177.0,12.487045,2.986793,6.167710,12.705712,13.782625,14.101339,15.651898
MinAbsEStateIndex,177.0,0.076271,0.064879,0.000297,0.026506,0.064606,0.107579,0.377372
MinEStateIndex,177.0,-1.573625,1.208344,-4.776651,-1.573089,-0.987440,-0.853454,-0.487332
qed,177.0,0.660573,0.209698,0.042501,0.593360,0.736495,0.797302,0.915960
...,...,...,...,...,...,...,...,...
fr_thiazole,177.0,0.022599,0.149042,0.000000,0.000000,0.000000,0.000000,1.000000
fr_thiocyan,177.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
fr_thiophene,177.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
fr_unbrch_alkane,177.0,0.056497,0.276289,0.000000,0.000000,0.000000,0.000000,2.000000


## Fingerprint

In [11]:
def calculate_morgan_fingerprint(df, smiles_column="Smile",
                                    radius=2, n_bits=2048 ):
    
    fp_gen=rdFingerprintGenerator.GetMorganGenerator(radius= radius, fpSize= n_bits)
    mols=df[smiles_column].apply(Chem.MolFromSmiles)
    fingerprints=[]
    for mol in mols:
        if mol is None:
            fingerPrints.append([np.nan] * n_bits)
            continue
        try:
          fp=fp_gen.GetFingerprint(mol)
          fp_bits=list(fp)
        except Exception:
           fp_bits=[np.nan]*(n_bits)
        
        fingerprints.append(fp_bits)
    
    dfp=pd.DataFrame(fingerprints, columns=[f"FP_{i}" for i in range(n_bits)])
    return dfp



In [12]:
fp=calculate_morgan_fingerprint(df)
fp

,FP_0,FP_1,FP_2,FP_3,FP_4,FP_5,FP_6,FP_7,FP_8,FP_9,...,FP_2038,FP_2039,FP_2040,FP_2041,FP_2042,FP_2043,FP_2044,FP_2045,FP_2046,FP_2047
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
173,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
174,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
175,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Fingerprint Information

In [ ]:
fp.shape

(177, 2048)

In [ ]:
fp.info()

<class 'pandas.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Columns: 2048 entries, FP_0 to FP_2047
dtypes: int64(2048)
memory usage: 2.8 MB


In [43]:
fp.describe().T

,count,mean,std,min,25%,50%,75%,max
FP_0,177.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
FP_1,177.0,0.288136,0.454179,0.0,0.0,0.0,1.0,1.0
FP_2,177.0,0.022599,0.149042,0.0,0.0,0.0,0.0,1.0
FP_3,177.0,0.011299,0.105996,0.0,0.0,0.0,0.0,1.0
FP_4,177.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
FP_2043,177.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
FP_2044,177.0,0.096045,0.295489,0.0,0.0,0.0,0.0,1.0
FP_2045,177.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
FP_2046,177.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


## Save Descriptors & Fingerprint DataFrame 

In [13]:
df_basic=dc.to_csv("../data/output/basic_descriptors.csv",  index=False)
df_extend= ds.to_csv("../data/output/extended_descriptors.csv",  index=False)
df_fingerprint= fp.to_csv("../data/output/fingerprint.csv",  index=False)